# Notebook 3: Academic Scholarship Analysis
How many scholarships exist? What are they worth? What's the probability a youth player receives one?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

schols = pd.read_csv('../data/processed/scholarship_outcomes.csv')
costs  = pd.read_csv('../data/processed/costs.csv')

In [ ]:
# --- 3A: Total scholarship pool by division & gender (2024) ---
s2024 = schols[schols['year'] == 2024].copy()

pivot = s2024.pivot_table(values='total_scholarship_pool_usd',
                          index='division', columns='gender', aggfunc='sum')
print('Total scholarship pool (USD) by division and gender (2024):')
print(pivot.applymap(lambda v: f'${v:,.0f}' if pd.notna(v) else '-'))

pivot.plot(kind='bar', color=['#e74c3c','#3498db'], alpha=0.85)
plt.title('Total Soccer Scholarship Pool by Division & Gender (2024)', fontsize=13)
plt.ylabel('USD')
plt.xticks(rotation=0)
plt.yaxis = plt.gca().yaxis
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'${v/1e6:.0f}M'))
plt.tight_layout()
plt.savefig('../data/analysis/scholarship_pool_by_div.png', dpi=150)
plt.show()

In [ ]:
# --- 3B: Scholarship probability for a given player ---
# Baseline: ~4 million recreational players → how many end up with any scholarship?

total_youth_players = 4_000_000  # estimate from AYSO + USYS

s2024_soccer = s2024[s2024['sport'].isin(['M Soccer','W Soccer'])].copy()
total_roster_spots = s2024_soccer['roster_spots_total'].sum()
total_schol_spots  = (s2024_soccer['roster_spots_total'] *
                      (s2024_soccer['partial_scholarship_pct'] +
                       s2024_soccer['full_scholarship_pct']) / 100).sum().round(0)

prob_any_college = total_roster_spots / total_youth_players * 100
prob_any_schol   = total_schol_spots  / total_youth_players * 100

print(f'Total youth soccer players (est): {total_youth_players:,}')
print(f'Total college soccer roster spots: {total_roster_spots:,.0f}')
print(f'Total spots with any scholarship:  {total_schol_spots:,.0f}')
print(f'Prob any player plays college soccer: {prob_any_college:.2f}%')
print(f'Prob any player gets a scholarship:   {prob_any_schol:.2f}%')

In [ ]:
# --- 3C: Average scholarship value by division ---
fig, axes = plt.subplots(1, 2, figsize=(14,5))

for ax, gender in zip(axes, ['Men', 'Women']):
    sub = s2024[s2024['gender'] == gender].sort_values('avg_scholarship_value_usd', ascending=False)
    ax.barh(sub['division'], sub['avg_scholarship_value_usd'], color='#3498db', alpha=0.85)
    ax.set_title(f'Avg Scholarship Value — {gender} Soccer (2024)', fontsize=12)
    ax.set_xlabel('USD')
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'${v:,.0f}'))

plt.tight_layout()
plt.savefig('../data/analysis/avg_scholarship_by_div.png', dpi=150)
plt.show()

In [ ]:
# --- 3D: Net cost after scholarship (cost - scholarship value) ---
# Scenario: Tier 3 player (ECNL) gets a D1 scholarship vs. no scholarship

# Career cost for Tier 3 mid estimate (7 years)
tier3_career_cost = costs[costs['tier'] == 3]['total_mid'].mean() * 7

# D1 scholarship (4 years)
d1_men_schol   = schols[(schols['division']=='D1') & (schols['gender']=='Men') & (schols['year']==2024)]['avg_scholarship_value_usd'].values[0] * 4
d1_women_schol = schols[(schols['division']=='D1') & (schols['gender']=='Women') & (schols['year']==2024)]['avg_scholarship_value_usd'].values[0] * 4

scenarios = {
    'Career cost (Tier 3)': tier3_career_cost,
    'D1 Men scholarship (4yr)': d1_men_schol,
    'D1 Women scholarship (4yr)': d1_women_schol,
    'Net: Men (cost - schol)': tier3_career_cost - d1_men_schol,
    'Net: Women (cost - schol)': tier3_career_cost - d1_women_schol,
}

for k, v in scenarios.items():
    print(f'  {k}: ${v:,.0f}')

pd.DataFrame.from_dict({'value': scenarios}, orient='columns').to_csv(
    '../data/analysis/net_cost_scholarship_scenarios.csv')